# Kapido Module Notes

This notebook summarizes the purpose of each major module in the project.

## 1) Backend Entry Module

- **`backend/app/main.py`**
  - Initializes FastAPI app.
  - Configures middleware (for example CORS).
  - Registers API routes (prediction, analytics, history).

## 2) Database Module

- **`backend/app/db/mongo.py`**
  - Holds MongoDB connection/client setup.
  - Provides reusable helpers for accessing the configured database/collection.
  - Centralizes DB connectivity concerns so route/service code stays clean.

## 3) API Data Models

- **`backend/app/models/schemas.py`**
  - Defines request/response schemas (Pydantic models).
  - Validates API payloads for prediction and history storage.
  - Enforces consistent data contracts across frontend and backend.

## 4) Route Layer

- **`backend/app/routes/prediction.py`**
  - Exposes HTTP endpoints (`/predict`, `/store-prediction`, `/history`, analytics routes).
  - Parses/validates incoming parameters.
  - Delegates business logic to service modules.

## 5) Service Layer

- **`backend/app/services/predictor.py`**
  - Loads model artifact and performs demand prediction inference.

- **`backend/app/services/preprocess.py`**
  - Handles feature preparation/transformation before inference.

- **`backend/app/services/history_service.py`**
  - Stores and fetches historical prediction records from MongoDB.

- **`backend/app/services/analytics_service.py`**
  - Computes aggregate insights (hourly, location, day-of-week, distribution, time-series).

- **`backend/app/services/insights_service.py`**
  - Converts numeric outputs into explainable severity labels and recommendations.

- **`backend/app/services/validation_service.py`**
  - Performs quality checks and model validation metrics (for example MAE/RMSE and anomaly checks).

## 6) Frontend App Modules

- **`frontend/app/page.tsx`**
  - Main input/prediction page for users.

- **`frontend/app/dashboard/page.tsx`**
  - Dashboard view for trends, history, and visual analytics.

- **`frontend/app/workflow/page.tsx`**
  - Displays end-to-end workflow explanation for the platform.

- **`frontend/components/*`**
  - Reusable UI building blocks (forms, maps, cards, charts, tooltips, diagrams).

- **`frontend/lib/api.ts`**
  - Central API client for backend calls.

- **`frontend/lib/types.ts`**
  - Shared TypeScript interfaces/types for API and UI data.

- **`frontend/lib/location-coordinates.ts`**
  - Location metadata used by forms/maps.

- **`frontend/lib/workflow-content.ts`**
  - Static content/config for workflow presentation.

## 7) ML Model Modules

- **`models/train_model.py`**
  - Trains the local/high-accuracy model profile and saves `model.pkl`.

- **`models/train_model_prod.py`**
  - Trains production-optimized profile and saves `model_prod.pkl` with size-aware trade-offs.

- **`models/predict.py`**
  - Lightweight script for local prediction checks against trained artifacts.

## 8) Testing Modules

- **`backend/tests/test_prediction_api.py`**
  - Verifies prediction endpoints and API behavior.

- **`backend/tests/test_insights_service.py`**
  - Verifies recommendation/severity logic in the insights service.

- **`backend/tests/conftest.py`**
  - Common test fixtures and setup utilities.

## 9) Notes for Extension

- Keep route handlers thin: put business logic in services.
- Keep schema changes synchronized between backend models and frontend types.
- Add tests whenever service logic or endpoint contracts are changed.
- Retrain models when feature columns or data distributions change.

## 10) How The Model Is Trained (End-to-End)

### Data Source and Features
- Training data comes from `data/ride_data.csv`.
- Core model inputs include:
  - `hour`
  - `day_of_week`
  - `location`
  - `latitude`
  - `longitude`
- Target variable is ride demand (used to estimate expected requests at a given context).

### Preprocessing and Feature Engineering
- Missing values are handled using imputation strategies (median/most-frequent depending on column type).
- Time/context features are standardized into model-ready columns.
- Categorical fields like `location` are encoded through the training pipeline.
- Numeric fields are scaled/transformed according to the model pipeline configuration.

### Training Profiles
- `models/train_model.py`
  - Trains the local/high-accuracy profile.
  - Prioritizes stronger predictive quality for local experimentation.
  - Outputs `models/model.pkl`.
- `models/train_model_prod.py`
  - Trains a production-optimized profile.
  - Balances model quality with artifact size and deployment constraints.
  - Outputs `models/model_prod.pkl` (compressed joblib artifact).

### Evaluation and Selection
- Training scripts compute metrics such as RMSE and MAE.
- Production script compares candidate models using an accuracy-vs-size trade-off.
- Final artifact is selected, serialized, and saved for backend inference.

### Output Artifacts
- Local artifact: `models/model.pkl`
- Production artifact: `models/model_prod.pkl`
- Backend selects artifact via `MODEL_PATH` in `backend/.env`.

## 11) How Future Data Is Predicted (Inference Flow)

### Step 1: New Input Arrives
- A user (or client app) sends future-context values to `GET /predict`, for example:
  - `hour=18`
  - `location=downtown`
  - `latitude=12.9716`
  - `longitude=77.5946`
- This represents a future or current demand scenario to estimate.

### Step 2: Request Validation
- Route layer validates query parameters and converts them into typed schema-compatible values.
- Invalid inputs are rejected early with descriptive API validation errors.

### Step 3: Feature Construction for Inference
- Service layer derives/normalizes required model features (including day/time context).
- Input is converted into the same feature structure used during training.
- This strict train-inference feature parity prevents schema drift issues.

### Step 4: Model Loading and Prediction
- Predictor service loads the model artifact from `MODEL_PATH` (`model.pkl` or `model_prod.pkl`).
- The exact same preprocessing pipeline stored in the artifact is applied to the new input row.
- Model outputs predicted demand value for that context.

### Step 5: Supply Estimation and Gap Calculation
- Backend estimates active driver supply for the same context.
- Gap is computed as:

$$
\text{Gap} = \text{Demand} - \text{Supply}
$$

- Severity level is derived from gap magnitude (for example low/medium/high shortage).

### Step 6: Explainability and Recommendations
- Insights service creates human-readable explanations and action recommendations.
- Example recommendation: move idle drivers toward shortage zones.

### Step 7: Optional Persistence and Analytics
- Prediction result can be saved via `POST /store-prediction`.
- Stored records power dashboard endpoints (`/history`, hourly trends, location trends, gap distribution, time-series).

### Example Prediction Response Shape
```json
{
  "demand": 123.45,
  "supply": 98.76,
  "gap": 24.69,
  "severity": "medium",
  "explanations": ["Demand currently exceeds active driver supply."],
  "recommendations": ["Reposition nearby idle drivers toward downtown."]
}
```

### Practical Rule
- Retrain and redeploy the model artifact whenever feature definitions or data distribution materially change, so future predictions remain calibrated.